# One-domain .nas mesh

In [13]:
# Read in data from swc file of a bifurcation and create vascularmd mesh for it
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pyvista as pv
from ArterialTree import ArterialTree

# ── Configuration ─────────────────────────────────────────────────────────────
N = 32       # nodes around each cross-section circumference (must be multiple of 8)
d = 0.08     # longitudinal cross-section density
layer_ratio = [0.16, 0.14, 0.7]  # O-grid radius ratio [a, b, c], a+b+c = 1
num_a, num_b = 1, 3            # number of O-grid layers in parts a and b

mesh_name = 'validation_geometry'
swc_path = os.path.join('..', 'Data', f'{mesh_name}.swc')
out_dir  = os.path.join('..', 'Output')
os.makedirs(out_dir, exist_ok=True)

# Build the vascularmd (ArterialTree) model straight from the SWC centerline data
tree = ArterialTree("patient", "Aneurisk", swc_path)
tree.model_network()                               # parametric modeling of the network
tree.compute_cross_sections(N, d, parallel=False)   # mesh cross sections

volume_mesh = tree.mesh_volume(layer_ratio, num_a, num_b)
print(f"Meshed full network: {volume_mesh.n_points} points, {volume_mesh.n_cells} cells")

# ── Write the volume mesh to Nastran (.nas), tagging every element as one domain ──
# Native route (no VTK/pyvista hop): get the mesh as a meshio.Mesh built straight
# from the tree's internal arrays, then attach a per-cell property id via
# cell_data["nastran:ref"]. meshio's Nastran writer emits that value into the
# CHEXA PID field (field 3, blank by default). Every hexahedron gets PID 1
# ("fluid"), so the whole mesh reads as one explicitly-tagged solid domain that
# COMSOL recognises as a single selection. Nastran PIDs are integers, so the name
# "fluid" is just our label for PID 1.
mesh = tree.get_volume_mesh("meshio")               # meshio.Mesh(points, [("hexahedron", conn)])
n_cells = mesh.cells[0].data.shape[0]
FLUID_PID = 1
mesh.cell_data["nastran:ref"] = [np.full(n_cells, FLUID_PID, dtype=int)]

nas_1domain_path = os.path.join(out_dir, f'{mesh_name}_volume_mesh_1domain.nas')
mesh.write(nas_1domain_path)
print(f"Saved one-domain Nastran mesh -> {os.path.abspath(nas_1domain_path)}")

# Generate a vtk mesh as well
# Same CHEXA elements, in VTK form: get_volume_mesh("pyvista") returns a
# pv.UnstructuredGrid built from the tree's internal arrays (node order matches the
# .nas above), which pyvista serialises directly to a legacy .vtk file. Handy for
# ParaView inspection and any pyvista-based post-processing.
vtk_mesh = tree.get_volume_mesh("pyvista")
vtk_path = os.path.join(out_dir, f'{mesh_name}_volume_mesh.vtk')
vtk_mesh.save(vtk_path)
print(f"Saved VTK volume mesh -> {os.path.abspath(vtk_path)} "
      f"({vtk_mesh.n_points} points, {vtk_mesh.n_cells} cells)")


Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing volume...
Meshed full network: 28938 points, 26880 cells
Saved one-domain Nastran mesh -> /home/xbfl0349/arterial_network/Output/validation_geometry_volume_mesh_1domain.nas
Saved VTK volume mesh -> /home/xbfl0349/arterial_network/Output/validation_geometry_volume_mesh.vtk (28938 points, 26880 cells)


# Two-domain .nas mesh

In [14]:
# Tag the layer_a (boundary) cells with PID 2 and everything else with PID 1.
#
# The volume mesh is a stack of "slabs": each cross-section pair contributes one
# hexahedron per 2-D O-grid quad, in the exact row order of ogrid_pattern_faces.
# So we (1) compute, per O-grid quad, whether it lies in part a (the radial band
# from the wall out to pb), (2) tile that flag across every slab, (3) write PID 2
# where layer_a else PID 1. Reuses the `tree` / mesh already built in the cell above.

# ── Stage 1: per-quad layer_a flag ────────────────────────────────────────────
# Mirror of ArterialTree.ogrid_pattern_faces (ArterialTree.py:2591), additionally
# recording each quad's radial index k (k = 0 at the wall, increasing inward -- the
# inner-loop variable at ArterialTree.py:2640). A quad is in part a when k <= num_a:
# lin_interp(surface, pb, num_a+2)[:-1] lays down num_a+1 intervals between the wall
# and pb, so the boundary band is num_a+1 cell layers thick. (Toggle to `k < num_a`
# for exactly num_a layers.)
def ogrid_faces_with_region(N, num_a, num_b):
    count = 0
    tot_faces = int(N/8) * (2 * (num_a + num_b + 2) + int(N/8)) * 4
    faces = np.zeros((tot_faces, 5), dtype=int)
    is_a  = np.zeros((tot_faces,), dtype=bool)
    shared_edge1 = []
    nb_faces = 0
    for s in range(4):
        j = 0
        for i in range(int(s * N/4), int((s + 1) * N/4)):
            if j <= int(N/8):
                if s != 0 and j == 0:
                    nb_vertices = int(num_a + num_b + 2)
                    ray_ind = list(range(count, count + nb_vertices)) + shared_edge1[::-1]
                    shared_edge1 = [ray_ind[-1]]
                elif s == 3:
                    nb_vertices = int(num_a + num_b + 2 + N/8)
                    ray_ind = list(range(count, count + nb_vertices)) + [first_ray[-j - 1]]
                else:
                    nb_vertices = int(num_a + num_b + 3 + N/8)
                    ray_ind = list(range(count, count + nb_vertices))
                    shared_edge1.append(ray_ind[-1])
                if j == int(N/8):
                    shared_edge2 = ray_ind[-int(N/8):]
            else:
                nb_vertices = num_a + num_b + 2
                ray_ind = list(range(count, count + nb_vertices)) + [shared_edge2[j - int(N/8) - 1]]
            if s == 0 and j == 0:
                first_ray = ray_ind
                ray_prec = ray_ind
            else:
                for k in range(min([len(ray_prec) - 1, len(ray_ind) - 1])):
                    faces[nb_faces, :] = np.array([4, ray_prec[k], ray_ind[k], ray_ind[k + 1], ray_prec[k + 1]])
                    is_a[nb_faces] = (k <= num_a)
                    nb_faces += 1
            count += nb_vertices
            ray_prec = ray_ind
            j = j + 1
    for k in range(min([len(ray_ind) - 1, len(first_ray) - 1])):
        faces[nb_faces, :] = np.array([4, ray_ind[k], first_ray[k], first_ray[k + 1], ray_ind[k + 1]])
        is_a[nb_faces] = (k <= num_a)
        nb_faces += 1
    return faces, is_a

faces, is_layer_a = ogrid_faces_with_region(N, num_a, num_b)
# Fidelity guard: the mirror must reproduce the library's own faces exactly,
# otherwise the radial-index labelling can't be trusted.
assert np.array_equal(faces, tree.ogrid_pattern_faces(N, num_a, num_b)), \
    "mirrored ogrid_pattern_faces diverged from ArterialTree.ogrid_pattern_faces"
nfaces = faces.shape[0]
print(f"O-grid quads per slab: {nfaces}  ->  {int(is_layer_a.sum())} layer_a, {int((~is_layer_a).sum())} rest")

# ── Stage 2: propagate the per-quad flag to every cell ────────────────────────
# Every slab writes exactly `nfaces` cells in this same quad order (vessel slabs
# and into-bifurcation slabs alike -- the bifurcation reorder only rewires the far
# side, not the row order/count). Validate against the link graph before tiling.
n_cells = tree.get_number_of_cells()
assert n_cells % nfaces == 0, f"cell count {n_cells} is not a multiple of {nfaces}"
n_slabs = n_cells // nfaces

link = tree.get_volume_link()                        # [edge_start, edge_end, crsec_index] per cell
link_slabs = link.reshape(n_slabs, nfaces, 3)
assert (link_slabs == link_slabs[:, :1, :]).all(), \
    "cells are not grouped into nfaces-sized slabs -- tiling would be misaligned"

cell_is_layer_a = np.tile(is_layer_a, n_slabs)
print(f"{n_cells} cells over {n_slabs} slabs -> "
      f"{int(cell_is_layer_a.sum())} layer_a cells (PID 2), {int((~cell_is_layer_a).sum())} rest (PID 1)")

# ── Stage 3: assign PID and write the two-domain .nas ─────────────────────────
LAYER_A_PID, REST_PID = 2, 1
pid = np.where(cell_is_layer_a, LAYER_A_PID, REST_PID).astype(int)

mesh = tree.get_volume_mesh("meshio")
mesh.cell_data["nastran:ref"] = [pid]                # -> CHEXA PID field (field 3)

nas_path = os.path.join(out_dir, f'{mesh_name}_volume_mesh_2domain.nas')
mesh.write(nas_path)
print(f"Saved two-domain Nastran mesh -> {os.path.abspath(nas_path)}")
print(f"  PID {LAYER_A_PID} = layer_a ({int(cell_is_layer_a.sum())} CHEXA), "
      f"PID {REST_PID} = rest ({int((~cell_is_layer_a).sum())} CHEXA)")

O-grid quads per slab: 256  ->  64 layer_a, 192 rest
26880 cells over 105 slabs -> 6720 layer_a cells (PID 2), 20160 rest (PID 1)
Saved two-domain Nastran mesh -> /home/xbfl0349/arterial_network/Output/validation_geometry_volume_mesh_2domain.nas
  PID 2 = layer_a (6720 CHEXA), PID 1 = rest (20160 CHEXA)


In [15]:
# ── Verify the two-domain tagging ─────────────────────────────────────────────
import meshio

# 1) Parse the .nas directly: PID is field 3 (cols 17-24) of each CHEXA card.
pid_counts = {}
with open(nas_path) as fh:
    for ln in fh:
        if ln.startswith("CHEXA"):
            p = ln[16:24].strip()
            pid_counts[p] = pid_counts.get(p, 0) + 1
print("CHEXA PID counts in file:", pid_counts)
assert pid_counts.get("2", 0) == int(cell_is_layer_a.sum())
assert pid_counts.get("1", 0) == int((~cell_is_layer_a).sum())

# 2) meshio round-trip: the tags must survive a read-back and match what we wrote.
rt = meshio.read(nas_path)
ref = rt.cell_data["nastran:ref"][0]
print("round-trip unique PIDs:", np.unique(ref),
      "| PID2:", int((ref == 2).sum()), " PID1:", int((ref == 1).sum()))
assert set(np.unique(ref).tolist()) == {1, 2}
assert np.array_equal(ref, pid)

# 3) Geometric sanity: layer_a (PID 2) cells should sit at the OUTER radius.
#    Per slab, measure each cell-centroid's distance from that slab's mean centroid
#    (~ the section axis); layer_a cells should have the larger radius.
cent = tree.get_volume_mesh().cell_centers().points          # same order as `pid`
cent_slabs = cent.reshape(n_slabs, nfaces, 3)
r = np.linalg.norm(cent_slabs - cent_slabs.mean(axis=1, keepdims=True), axis=2)
is_a_slab = np.tile(is_layer_a, (n_slabs, 1))
print(f"median centroid radius:  layer_a = {np.median(r[is_a_slab]):.3f}   "
      f"rest = {np.median(r[~is_a_slab]):.3f}   (layer_a should be larger)")
assert np.median(r[is_a_slab]) > np.median(r[~is_a_slab])
print("All checks passed.")


CHEXA PID counts in file: {'2': 6720, '1': 20160}
round-trip unique PIDs: [1 2] | PID2: 6720  PID1: 20160
median centroid radius:  layer_a = 5.494   rest = 4.251   (layer_a should be larger)
All checks passed.


# Inlet & outlet tags

In [16]:
# Stage 1-2 (plan_for_nas_boundary_tagging.md): tag the INLET and OUTLET fluid
# cross-sections as their own boundary selections in the .nas, at generation time,
# using only native vascularmd functions.
#
# Mechanism: a boundary face becomes a COMSOL boundary selection when it is emitted
# as a coincident shell element (CQUAD4) carrying its own property id (PID), reusing
# the hexahedron's corner node IDs. meshio's Nastran writer maps a "quad" cell block
# -> CQUAD4 and writes cell_data["nastran:ref"] into the PID field, sharing one global
# element-ID sequence with the CHEXA block. We emit ONE quad block for all terminal
# caps and give each quad its own PID (inlet vs outlet).
#
# Native terminal identification (no geometric tolerance):
#   * Inlet  = model-graph "end" node with in-degree 0; outlets = the other "end"
#     nodes -- the exact split ArterialTree.get_inlet_outlet() uses.
#   * A terminal's cap is the O-grid disk of its cross-section, reconstructed exactly
#     as ogrid_pattern_faces(N, num_a, num_b) offset by that section's base vertex id
#     (crsec_graph.nodes[node]['id']) -- the same bookkeeping mesh_volume uses.
#   * FLUID-ONLY: keep just the fluid (non-layer-a) quads via `~is_layer_a` (row-aligned
#     with ogrid_pattern_faces), so the flow BC sits on the fluid core, not the wall
#     annulus. The excluded outer annulus is the wall-end rim (group B5), a later stage.
# Reuses `tree`, `N`, `num_a`, `num_b`, `pid`, `is_layer_a`, `n_cells`, `out_dir`,
# `mesh_name` from the cells above.
import meshio

INLET_PID, OUTLET_PID = 111, 131   # boundary PIDs (distinct from cell PIDs 1/2)

model_graph = tree.get_model_graph()
crsec_graph = tree.get_crsec_graph()
inlet_nodes  = [n for n in model_graph.nodes()
                if model_graph.nodes[n]['type'] == 'end' and model_graph.in_degree(n) == 0]
outlet_nodes = [n for n in model_graph.nodes()
                if model_graph.nodes[n]['type'] == 'end' and model_graph.in_degree(n) != 0]
assert len(inlet_nodes) == 1, f"expected exactly one inlet, found {len(inlet_nodes)}"
assert len(outlet_nodes) >= 1, "expected at least one outlet"
fluid_quad = ~is_layer_a                                    # row-aligned with ogrid_pattern_faces

def fluid_cap(node):
    """Fluid-only O-grid disk of a terminal cross-section, in volume node IDs."""
    base = crsec_graph.nodes[node]['id']                   # base offset of this section's O-grid vertices
    return (tree.ogrid_pattern_faces(N, num_a, num_b)[:, 1:] + base)[fluid_quad]

# ── Build one quad block: inlet caps (PID 111) + outlet caps (PID 131) ─────────
quad_blocks, quad_pids = [], []
for n in inlet_nodes:
    c = fluid_cap(n); quad_blocks.append(c); quad_pids.append(np.full(len(c), INLET_PID, int))
for n in outlet_nodes:
    c = fluid_cap(n); quad_blocks.append(c); quad_pids.append(np.full(len(c), OUTLET_PID, int))
quad_conn = np.vstack(quad_blocks)
quad_pid  = np.concatenate(quad_pids)
n_in  = int((quad_pid == INLET_PID).sum())
n_out = int((quad_pid == OUTLET_PID).sum())

# ── Assemble hexahedra (domain PIDs) + terminal fluid caps (boundary PIDs) ─────
base_mesh = tree.get_volume_mesh("meshio")
tagged = meshio.Mesh(base_mesh.points,
                     [("hexahedron", base_mesh.cells[0].data), ("quad", quad_conn)])
tagged.cell_data["nastran:ref"] = [pid, quad_pid]          # CHEXA PIDs 1/2 ; CQUAD4 PIDs 111/131

nas_inlet_path = os.path.join(out_dir, f'{mesh_name}_inlet_tagged.nas')
tagged.write(nas_inlet_path)
print(f"Saved inlet/outlet-tagged Nastran mesh -> {os.path.abspath(nas_inlet_path)}")
print(f"  {n_cells} CHEXA (domains 1/2) + {n_in} CQUAD4 inlet (PID {INLET_PID}) "
      f"+ {n_out} CQUAD4 outlet (PID {OUTLET_PID}) over {len(outlet_nodes)} outlet(s)")

# ── Verify (acceptance check) ─────────────────────────────────────────────────
# 1) File: per-PID CQUAD4 counts, CHEXA left intact.
from collections import Counter
cq = Counter(); n_ch = 0
with open(nas_inlet_path) as fh:
    for ln in fh:
        if ln.startswith("CQUAD4"): cq[ln[16:24].strip()] += 1
        elif ln.startswith("CHEXA"): n_ch += 1
print(f"file: CHEXA={n_ch}  CQUAD4 by PID={dict(cq)}")
assert n_ch == n_cells
assert cq[str(INLET_PID)] == n_in and cq[str(OUTLET_PID)] == n_out

# 2) meshio round-trip: shell PIDs survive read-back.
rt = meshio.read(nas_inlet_path)
qi = [i for i, cb in enumerate(rt.cells) if cb.type == "quad"][0]
assert set(np.unique(rt.cell_data["nastran:ref"][qi]).tolist()) == {INLET_PID, OUTLET_PID}

# 3) Fluid-only: no tagged node belongs to a wall-only (layer_a) cap quad.
all_cap0  = tree.ogrid_pattern_faces(N, num_a, num_b)[:, 1:]
wall_only = np.setdiff1d(np.unique(all_cap0[is_layer_a]), np.unique(all_cap0[fluid_quad]))  # local ids
for node in inlet_nodes + outlet_nodes:
    b = crsec_graph.nodes[node]['id']
    assert not np.intersect1d(np.unique(fluid_cap(node)), wall_only + b).size, \
        f"terminal {node} cap still contains wall-interior nodes"

# 4) Native geometric sanity: each terminal cap is planar and matches a
#    get_inlet_outlet centre/tangent of the right kind.
terms = tree.get_inlet_outlet()
def check_planar(block, label):
    P = base_mesh.points[np.unique(block)]
    cands = [(np.asarray(c, float)[:3], np.asarray(t, float)[:3]) for c, t, l in terms if l == label]
    c, t = min(cands, key=lambda ct: np.linalg.norm(ct[0] - P.mean(0)))
    t = t / np.linalg.norm(t)
    R = np.linalg.norm(P - c, axis=1).max()
    resid = np.abs((P - c) @ t).max()
    assert resid < 0.02 * R, f"{label} cap not planar (resid {resid:.2e}, R {R:.3f})"
    return R, resid
for n in inlet_nodes:  R_in, e_in = check_planar(fluid_cap(n), "inlet")
for n in outlet_nodes: check_planar(fluid_cap(n), "outlet")
print(f"planarity ok (inlet fluid radius {R_in:.3f}, residual {e_in:.1e}); outlets matched to outlet terminals.")
print("Checks passed: inlet (PID 111) and outlet (PID 131) are distinct fluid boundary selections.\n"
      "COMSOL import note: turn OFF 'Allow partitioning of shells' to keep each as one selection.")

Saved inlet/outlet-tagged Nastran mesh -> /home/xbfl0349/arterial_network/Output/validation_geometry_inlet_tagged.nas
  26880 CHEXA (domains 1/2) + 192 CQUAD4 inlet (PID 111) + 192 CQUAD4 outlet (PID 131) over 1 outlet(s)
file: CHEXA=26880  CQUAD4 by PID={'111': 192, '131': 192}
planarity ok (inlet fluid radius 5.040, residual 2.1e-04); outlets matched to outlet terminals.
Checks passed: inlet (PID 111) and outlet (PID 131) are distinct fluid boundary selections.
COMSOL import note: turn OFF 'Allow partitioning of shells' to keep each as one selection.


# Wall, interface & annulus tags (complete FSI boundary set)

In [17]:
# Stage 3 (plan_for_nas_boundary_tagging.md): add the three remaining boundary
# groups and write ONE .nas carrying the complete FSI tag set:
#   PID 1 / 2   CHEXA domains          fluid (b+c)  /  wall (layer a)
#   PID 111     inlet  fluid cap                     (velocity BC)
#   PID 131     outlet fluid cap(s)                  (pressure BC)
#   PID 221     wall-end annulus rings at terminals  (B5, wall Fixed Constraint)
#   PID 101     fluid<->wall interface (a|b surface)  (B1, FSI coupling; interior)
#   PID 201     outer wall surface (lumen surface)    (B2, free / external pressure)
#
# Native extraction, no tolerances:
#   * annulus(node) = the wall (layer_a) quads of that terminal's O-grid disk -- the
#     exact complement of the fluid cap tagged in the previous cell.
#   * interface = faces shared by the two domains: split the mesh by cell PID and
#     intersect the wall-region boundary with the fluid-region boundary (pyvista
#     threshold + extract_surface, original node ids carried through a 'vid' array).
#     It is interior, hence disjoint from every boundary group.
#   * outer wall = every boundary face that is NOT a terminal cap (fluid cap OR
#     annulus). This is precisely layer_a's outer surface, so it automatically
#     EXCLUDES the inlet/outlet caps (111/131) and the annulus rings (221).
# Reuses tree, N, num_a, num_b, pid, is_layer_a, fluid_quad, crsec_graph,
# inlet_nodes, outlet_nodes, INLET_PID, OUTLET_PID, base_mesh, n_cells, out_dir,
# mesh_name from the cells above.

IFACE_PID, OUTER_PID, ANNULUS_PID = 101, 201, 221
of = tree.ogrid_pattern_faces(N, num_a, num_b)[:, 1:]        # local O-grid quad connectivity
def full_cap(node):
    """The whole O-grid disk (fluid core + wall annulus) of a terminal, in volume node IDs."""
    return of + crsec_graph.nodes[node]['id']
terminals = inlet_nodes + outlet_nodes

# Volume mesh as pyvista, carrying original point ids ('vid') and the domain tag ('dom').
grid = tree.get_volume_mesh("pyvista")
grid.point_data['vid'] = np.arange(grid.n_points)
grid.cell_data['dom']  = pid
def surf_conn(surf):
    """Boundary quads of a (sub)mesh, expressed in ORIGINAL volume node IDs."""
    return surf.point_data['vid'][surf.faces.reshape(-1, 5)[:, 1:]]
def keyset(conn):
    return set(map(frozenset, conn.tolist()))

# ── Boundary faces of the whole mesh ──────────────────────────────────────────
boundary = surf_conn(grid.extract_surface())

# ── Terminal caps: fluid (111/131) reused split + wall annulus (221) ───────────
inlet_fc  = [full_cap(n)[fluid_quad] for n in inlet_nodes]
outlet_fc = [full_cap(n)[fluid_quad] for n in outlet_nodes]
annulus   = np.vstack([full_cap(n)[is_layer_a] for n in terminals])   # wall-end rims
cap_keys  = keyset(np.vstack(inlet_fc + outlet_fc + [annulus]))       # every terminal-cap face

# ── Outer wall (201) = boundary minus every terminal cap ──────────────────────
outer = np.array([q for q in boundary if frozenset(q.tolist()) not in cap_keys])

# ── Interface (101) = wall-region boundary ∩ fluid-region boundary ────────────
wall_b  = surf_conn(grid.threshold([2, 2], scalars='dom').extract_surface())
fluid_b = surf_conn(grid.threshold([1, 1], scalars='dom').extract_surface())
iface_keys = keyset(wall_b) & keyset(fluid_b)
interface  = np.array([q for q in wall_b if frozenset(q.tolist()) in iface_keys])

# ── Assemble one quad block with all five boundary groups ─────────────────────
groups = [(np.vstack(inlet_fc), INLET_PID), (np.vstack(outlet_fc), OUTLET_PID),
          (annulus, ANNULUS_PID), (interface, IFACE_PID), (outer, OUTER_PID)]
quad_conn = np.vstack([g for g, _ in groups])
quad_pid  = np.concatenate([np.full(len(g), p, int) for g, p in groups])

tagged = meshio.Mesh(base_mesh.points,
                     [("hexahedron", base_mesh.cells[0].data), ("quad", quad_conn)])
tagged.cell_data["nastran:ref"] = [pid, quad_pid]
nas_fsi_path = os.path.join(out_dir, f'{mesh_name}_fsi_tagged.nas')
tagged.write(nas_fsi_path)
counts = {p: int((quad_pid == p).sum()) for _, p in groups}
print(f"Saved full FSI-tagged Nastran mesh -> {os.path.abspath(nas_fsi_path)}")
print(f"  {n_cells} CHEXA (domains 1/2) + {len(quad_conn)} CQUAD4")
print(f"  inlet(111)={counts[111]}  outlet(131)={counts[131]}  annulus(221)={counts[221]}  "
      f"interface(101)={counts[101]}  outer(201)={counts[201]}")

# ── Verify ────────────────────────────────────────────────────────────────────
# 1) Boundary partitions exactly into {outer, fluid caps, annulus}; interface is interior.
bkeys = keyset(boundary)
assert keyset(outer) | cap_keys == bkeys, "outer + caps must cover every boundary face"
assert keyset(outer).isdisjoint(cap_keys), "outer must exclude the caps (111/131/221)"
assert keyset(interface).isdisjoint(bkeys), "interface must be interior (not on the outer shell)"
assert len(outer) + len(cap_keys) == len(bkeys)

# 2) File: per-PID CQUAD4 counts and untouched CHEXA.
from collections import Counter
cq = Counter(); n_ch = 0
with open(nas_fsi_path) as fh:
    for ln in fh:
        if ln.startswith("CQUAD4"): cq[ln[16:24].strip()] += 1
        elif ln.startswith("CHEXA"): n_ch += 1
assert n_ch == n_cells
assert {int(k): v for k, v in cq.items()} == counts

# 3) meshio round-trip: all five shell PIDs survive.
rt = meshio.read(nas_fsi_path)
qi = [i for i, cb in enumerate(rt.cells) if cb.type == "quad"][0]
assert set(np.unique(rt.cell_data["nastran:ref"][qi]).tolist()) == {101, 111, 131, 201, 221}

# 4) Radial sanity at the inlet: the annulus ring (221) sits OUTSIDE the fluid cap (111).
c = np.asarray([b for b in tree.get_inlet_outlet() if b[2] == "inlet"][0][0], float)[:3]
ri = inlet_nodes[0]
r_fluid = np.median(np.linalg.norm(base_mesh.points[np.unique(full_cap(ri)[fluid_quad])] - c, axis=1))
r_ann   = np.median(np.linalg.norm(base_mesh.points[np.unique(full_cap(ri)[is_layer_a])]  - c, axis=1))
print(f"inlet median radius: fluid cap = {r_fluid:.3f}, annulus = {r_ann:.3f} (annulus outer = {r_ann > r_fluid})")
assert r_ann > r_fluid
print("Checks passed: 2 domains + 5 boundary selections (101/111/131/201/221), partitioned and disjoint.\n"
      "COMSOL import note: turn OFF 'Allow partitioning of shells' to keep each group as one selection.")

Saved full FSI-tagged Nastran mesh -> /home/xbfl0349/arterial_network/Output/validation_geometry_fsi_tagged.nas
  26880 CHEXA (domains 1/2) + 7232 CQUAD4
  inlet(111)=192  outlet(131)=192  annulus(221)=128  interface(101)=3360  outer(201)=3360
inlet median radius: fluid cap = 4.232, annulus = 5.520 (annulus outer = True)
Checks passed: 2 domains + 5 boundary selections (101/111/131/201/221), partitioned and disjoint.
COMSOL import note: turn OFF 'Allow partitioning of shells' to keep each group as one selection.


# Scaled Jacobian

In [18]:
# ── Check the final mesh: scaled Jacobian of every hexahedral element ─────────
# Same metric the native ArterialTree.check_mesh() uses (compute_cell_quality
# ('scaled_jacobian'), ArterialTree.py:4615), but applied directly to `grid` --
# the exact CHEXA elements written to the .nas files above. check_mesh() itself
# is not called because it re-meshes each edge/bifurcation with mesh_volume's
# DEFAULT O-grid parameters (layer_ratio=[0.2,0.4,0.4], num_a=4, num_b=4;
# ArterialTree.py:2003), i.e. it would grade a different mesh than the one
# configured in this notebook.
# Scaled Jacobian per hex: 1 = perfect cube, -> 0 = collapsing, <= 0 = inverted.
THRES = 0.0                     # failure threshold (check_mesh's default `thres`)

q = grid.compute_cell_quality('scaled_jacobian')['CellQuality']  # order matches `pid`/`link`
assert len(q) == n_cells

print(f"scaled Jacobian over {n_cells} CHEXA:")
print(f"  overall:        min = {q.min():.4f}   mean = {q.mean():.4f}   max = {q.max():.4f}")
for dom, name in [(REST_PID, "fluid (PID 1)"), (LAYER_A_PID, "wall  (PID 2)")]:
    qd = q[pid == dom]
    print(f"  {name}:  min = {qd.min():.4f}   mean = {qd.mean():.4f}   ({len(qd)} cells)")
qs = np.quantile(q, [0.0, 0.01, 0.05, 0.25, 0.50, 1.0])
print("  quantiles 0/1/5/25/50/100 %:  " + "  ".join(f"{v:.3f}" for v in qs))

# Localize any failures: `link` gives [edge_start, edge_end, crsec_index] per cell.
failed = np.flatnonzero(q <= THRES)
if failed.size:
    print(f"\nFAILED: {failed.size} cells with scaled Jacobian <= {THRES} (worst 10 below)")
    for i in failed[np.argsort(q[failed])][:10]:
        e0, e1, cr = link[i]
        print(f"  cell {i:>7}  q = {q[i]:+.4f}  edge ({e0},{e1})  cross-section {cr}")
    print("  affected edges:", sorted({tuple(link[i, :2]) for i in failed}))
else:
    print(f"\nAll {n_cells} elements pass: min scaled Jacobian {q.min():.4f} > {THRES}.")

assert failed.size == 0, (f"{failed.size} degenerate/inverted hexahedra "
                          f"(scaled Jacobian <= {THRES}) -- fix before exporting to COMSOL")


scaled Jacobian over 26880 CHEXA:
  overall:        min = 0.7071   mean = 0.9570   max = 0.9951
  fluid (PID 1):  min = 0.7071   mean = 0.9446   (20160 cells)
  wall  (PID 2):  min = 0.9934   mean = 0.9943   (6720 cells)
  quantiles 0/1/5/25/50/100 %:  0.707  0.707  0.846  0.942  0.987  0.995

All 26880 elements pass: min scaled Jacobian 0.7071 > 0.0.
